## Feature Preparation for First-Phase Structural Prediction

This notebook prepares composition-based features for predicting
first-phase structural parameters: `a_1`, `b_1`, `c_1`, and `Grain_1`.

### Workflow

1. Load compound formulas, structural targets, and elemental descriptors.
2. Weight descriptors by stoichiometric coefficients and sum metal contributions.
3. Export 25 input features together with four structural targets.

Formulas must contain one or two metals followed by one nonmetal.
Stoichiometric coefficients are used without normalization.
Run cells sequentially.

In [1]:
# ============================================================
# 1. Import dependencies
# ============================================================

import re
import json

import numpy as np
import pandas as pd

In [2]:
# ============================================================
# 2. Load compounds, structural targets, and descriptors
# ============================================================

# Select rows with index labels 0 through 160, inclusive.
# The four target columns describe the first phase.
tmcs = pd.read_excel(
    "data/6_Structural_Pure.xlsx",
    sheet_name="TMCs",
    usecols=[
        "compounds",
        "a_1",
        "b_1",
        "c_1",
        "Grain_1",
    ],
).loc[:160, :]

print("Compound table shape:", tmcs.shape)

# Specify the element identifier and 13 metal descriptors.
cols_tm = [
    "element",
    "AN",
    "AW",
    "CR",
    "AR",
    "FIE",
    "SIE",
    "EA",
    "EN",
    "Nd",
    "Nv",
    "D",
    "EC",
    "TC",
]

# Replace missing metal descriptor values and "/" placeholders with zero.
tm = (
    pd.read_excel(
        "data/6_Structural_Pure.xlsx",
        sheet_name="TM_descriptors",
        usecols=cols_tm,
    )
    .fillna(0, inplace=False)
    .replace("/", 0, inplace=False)
)

# Specify the element identifier and 12 nonmetal descriptors.
cols_c = [
    "element",
    "AN",
    "AW",
    "CR",
    "AR",
    "VR",
    "FIE",
    "SIE",
    "EA",
    "EN",
    "Np",
    "Nv",
    "D",
]

# Replace missing nonmetal descriptor values with zero.
c = (
    pd.read_excel(
        "data/6_Structural_Pure.xlsx",
        sheet_name="C_descriptors",
        usecols=cols_c,
    )
    .fillna(0, inplace=False)
)


Compound table shape: (160, 5)


In [3]:
# ============================================================
# 3. Define chemical formula parsing functions
# ============================================================

def parse_compound(compound):
    """Extract element symbols and coefficients from a simple formula.

    Parameters
    ----------
    compound : str
        Chemical formula containing element symbols and optional
        integer or decimal stoichiometric coefficients.

    Returns
    -------
    dict
        Element symbols mapped to their stoichiometric coefficients,
        preserving their order of appearance.

    Notes
    -----
    An omitted coefficient is interpreted as 1.0.
    This parser is intended for simple formulas without parentheses.
    Repeated element symbols overwrite earlier values.
    """
    matches = re.findall(
        r"([A-Z][a-z]*)(\d*\.*\d*)",
        compound,
    )

    result = {}

    for match in matches:
        element, count = match

        # Use a coefficient of 1.0 when no number is provided.
        count = float(count) if count else 1.0
        result[element] = count

    return result


def get_res(compound_name):
    """Return the parsed elemental composition of a compound formula."""
    parsed_result = parse_compound(compound_name)
    return parsed_result


# Check formula parsing using a representative bimetallic nitride.
test = get_res("Mn0.2Cr0.8N")


In [4]:
# ============================================================
# 4. Inspect descriptor columns and the output schema
# ============================================================

display(tm.columns.tolist())
display(c.columns.tolist())

# Define the descriptor and target column names for inspection.
# Descriptor order follows the loaded descriptor tables.
columns = (
    [f"AB_{i}" for i in tm.columns.tolist()[1:]]
    + [f"C_{i}" for i in c.columns.tolist()[1:]]
    + [
        "a_1",
        "b_1",
        "c_1",
        "Grain_1",
    ]
)

display(np.array(columns))

print(
    "Compound column data type:",
    tmcs["compounds"].dtype,
)



['element',
 'AN',
 'AW',
 'CR',
 'AR',
 'FIE',
 'SIE',
 'EA',
 'EN',
 'Nd',
 'Nv',
 'D',
 'EC',
 'TC']

['element',
 'AN',
 'AW',
 'CR',
 'AR',
 'VR',
 'FIE',
 'SIE',
 'EA',
 'EN',
 'Np',
 'Nv',
 'D']

array(['AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE', 'AB_EA',
       'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN',
       'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN',
       'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1'], dtype='<U7')

Compound column data type: object


In [5]:
# ============================================================
# 5. Construct composition-based features
# ============================================================

train_data = []

# Align positional iteration with label-based access to target values.
tmcs = tmcs.reset_index(drop=True)

for index, formula in enumerate(tmcs["compounds"]):

    compound_feat = [formula]

    composition = get_res(formula)

    keys = list(composition.keys())
    values = list(composition.values())

    # Expected order for a ternary formula:
    # first metal, second metal, and nonmetal.
    if len(keys) > 2:

        A_key = keys[0]
        B_key = keys[1]
        C_key = keys[2]

        A_index = float(values[0])
        B_index = float(values[1])
        C_index = float(values[2])

        # Retrieve the first metal's descriptors, excluding its symbol,
        # and multiply them by its stoichiometric coefficient.
        A_feat = (
            tm[tm["element"] == A_key]
            .values
            .flatten()[1:]
            .astype(float)
            * A_index
        )

        # Retrieve and weight the second metal's descriptors.
        B_feat = (
            tm[tm["element"] == B_key]
            .values
            .flatten()[1:]
            .astype(float)
            * B_index
        )

        # Retrieve and weight the nonmetal descriptors.
        C_feat = (
            c[c["element"] == C_key]
            .values
            .flatten()[1:]
            .astype(float)
            * C_index
        )

        # Sum metal contributions without normalizing their coefficients.
        AB_feat = A_feat + B_feat

    else:

        # Expected order for a binary formula: metal and nonmetal.
        AB_key = keys[0]
        C_key = keys[1]

        AB_index = float(values[0])
        C_index = float(values[1])

        # Weight the single metal's descriptors by its formula coefficient.
        AB_feat = (
            tm[tm["element"] == AB_key]
            .values
            .flatten()[1:]
            .astype(float)
            * AB_index
        )

        # Weight the nonmetal descriptors by its formula coefficient.
        C_feat = (
            c[c["element"] == C_key]
            .values
            .flatten()[1:]
            .astype(float)
            * C_index
        )

    # Append 13 metal features, 12 nonmetal features, and four targets.
    # Structural targets are retained as provided in the source table.
    compound_feat += (
        list(AB_feat)
        + list(C_feat)
        + [
            tmcs.loc[index, "a_1"],
            tmcs.loc[index, "b_1"],
            tmcs.loc[index, "c_1"],
            tmcs.loc[index, "Grain_1"],
        ]
    )

    train_data.append(compound_feat)

In [6]:
# ============================================================
# 6. Assemble and export the regression dataset
# ============================================================

# Use AB_ for the combined metal descriptors and C_ for the nonmetal.
columns = (
    [f"AB_{column}" for column in tm.columns[1:]]
    + [f"C_{column}" for column in c.columns[1:]]
    + [
        "a_1",
        "b_1",
        "c_1",
        "Grain_1",
    ]
)

colnames = ["compound"] + columns

train_data_ = pd.DataFrame(
    train_data,
    columns=colnames,
)

train_data_.to_csv(
    "data/compound_6_Structural_pure.csv",
    index=False,
)

print("Data shape:", train_data_.shape)

# Exclude the compound identifier and all four structural targets.
print("Feature number:", train_data_.shape[1] - 5)

display(train_data_.head())


Data shape: (160, 30)
Feature number: 25


,compound,AB_AN,AB_AW,AB_CR,AB_AR,AB_FIE,AB_SIE,AB_EA,AB_EN,AB_Nd,...,C_SIE,C_EA,C_EN,C_Np,C_Nv,C_D,a_1,b_1,c_1,Grain_1
0,CoS2,27.0,58.933194,1.260,1.520,7.8810,17.0800,0.6600,1.880,7.0,...,46.68,4.14,5.16,8.0,12.0,3.92,5.5245,5.5245,5.5245,22.8
1,Cr0.1Co0.9S2,26.7,58.239485,1.261,1.534,7.7696,16.9908,0.6606,1.858,6.7,...,46.68,4.14,5.16,8.0,12.0,3.92,5.5236,5.5236,5.5236,25.6
2,Cr0.2Co0.8S2,26.4,57.545775,1.262,1.548,7.6582,16.9016,0.6612,1.836,6.4,...,46.68,4.14,5.16,8.0,12.0,3.92,5.5245,5.5245,5.5245,26.2
3,Cr0.5Co0.5S2,25.5,55.464647,1.265,1.590,7.3240,16.6340,0.6630,1.770,5.5,...,46.68,4.14,5.16,8.0,12.0,3.92,5.5262,5.5262,5.5262,20.2
4,Fe0.1Co0.9S2,26.9,58.624375,1.259,1.524,7.8832,17.1889,0.6103,1.875,6.9,...,46.68,4.14,5.16,8.0,12.0,3.92,5.5290,5.5290,5.5290,24.3


In [7]:
# ============================================================
# 7. Inspect the final output columns
# ============================================================

display(train_data_.columns)

Index(['compound', 'AB_AN', 'AB_AW', 'AB_CR', 'AB_AR', 'AB_FIE', 'AB_SIE',
       'AB_EA', 'AB_EN', 'AB_Nd', 'AB_Nv', 'AB_D', 'AB_EC', 'AB_TC', 'C_AN',
       'C_AW', 'C_CR', 'C_AR', 'C_VR', 'C_FIE', 'C_SIE', 'C_EA', 'C_EN',
       'C_Np', 'C_Nv', 'C_D', 'a_1', 'b_1', 'c_1', 'Grain_1'],
      dtype='object')